In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6" 

import sys
sys.path.append('../')
from helpers import *
# os.environ["XLA_FLAGS"] = "--xla_gpu_enable_cudnn_frontend=false"
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import jax
from jax import random, grad, jit, vmap
from jax import config
import jax.numpy as np
from jax.example_libraries import stax
from jax.example_libraries import optimizers

from livelossplot import PlotLosses
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm as tqdm
import numpy as onp

import pickle




jax.devices()        # list available devices
jax.default_backend()  # e.g. 'cpu', 'gpu'

rng = random.PRNGKey(0)

filename = 'lego_400.npz'
if not os.path.exists(filename):
    !gdown --id 108jNfjPITTsTA0lE6Kpg7Ei53BUVL-4n # Lego

data = np.load(filename)
images = data['images']
poses = data['poses']
focal = float(data['focal'])
H, W = images.shape[1:3]

images, val_images, test_images = np.split(images[...,:3], [100,107], axis=0)
poses, val_poses, test_poses = np.split(poses, [100,107], axis=0)

TEST_IDX = 10

print(val_images.shape, test_images.shape, focal)
plt.imshow(test_images[TEST_IDX,...])
plt.show()

In [ ]:
def get_rays(H, W, focal, c2w):
    i, j = np.meshgrid(np.arange(W), np.arange(H), indexing='xy')
    dirs = np.stack([(i-W*.5)/focal, -(j-H*.5)/focal, -np.ones_like(i)], -1)
    rays_d = np.sum(dirs[..., np.newaxis, :] * c2w[:3,:3], -1)
    rays_o = np.broadcast_to(c2w[:3,-1], rays_d.shape)
    return np.stack([rays_o, rays_d], 0)

get_rays = jit(get_rays, static_argnums=(0, 1, 2,))

training_rays = np.stack([get_rays(H,W,focal,pose) for pose in poses], 1)
training_data = np.concatenate([training_rays, images[None]])
training_data = np.moveaxis(training_data, 0, -2)
training_data = onp.array(np.reshape(training_data, [-1, 3, 3]))
onp.random.shuffle(training_data)
training_data = np.array(training_data)

In [ ]:
def render_rays(apply_fn, params, avals, bvals, key, rays, near, far, N_samples, rand=False, allret=False):
    rays_o, rays_d = rays
    
    # Compute 3D query points
    z_vals = np.linspace(near, far, N_samples) 
    if rand:
        z_vals += random.uniform(key, shape=list(rays_o.shape[:-1]) + [N_samples]) * (far-near)/N_samples
    pts = rays_o[...,None,:] + rays_d[...,None,:] * z_vals[...,:,None]
    
    # Run network
    pts_flat = np.reshape(pts, [-1,3])
    if avals is not None:
        pts_flat = np.concatenate([avals * np.sin(pts_flat @ bvals.T), 
                                   avals * np.cos(pts_flat @ bvals.T)], axis=-1)
    raw = apply_fn(params, pts_flat)
    raw = np.reshape(raw, list(pts.shape[:-1]) + [4])
    
    # Compute opacities and colors
    rgb, sigma_a = raw[...,:3], raw[...,3]
    sigma_a = jax.nn.relu(sigma_a)
    rgb = jax.nn.sigmoid(rgb) 
    
    # Do volume rendering
    dists = np.concatenate([z_vals[..., 1:] - z_vals[..., :-1], np.broadcast_to(np.array([1e10]), z_vals[...,:1].shape)], -1) 
    alpha = 1.-np.exp(-sigma_a * dists)
    trans = np.minimum(1., 1.-alpha + 1e-10)
    trans = np.concatenate([np.ones_like(trans[...,:1]), trans[...,:-1]], -1)  
    weights = alpha * np.cumprod(trans, -1)
    
    rgb_map = np.sum(weights[...,None] * rgb, -2) 
    acc_map = np.sum(weights, -1)
    
    if False:
        rgb_map = rgb_map + (1.-acc_map[..., None])
    
    if not allret:
        return rgb_map
    
    depth_map = np.sum(weights * z_vals, -1) 

    return rgb_map, depth_map, acc_map

def render_fn_inner(params, avals, bvals, key, rays, rand, allret):
    return render_rays(apply_fn, params, avals, bvals, key, rays, near=2., far=6., N_samples=N_samples, rand=rand, allret=allret)
render_fn_inner = jit(render_fn_inner, static_argnums=(5, 6,))

def render_fn(params, avals, bvals, key, rays, rand):
    chunk = 5
    for i in range(0, rays.shape[1], chunk):
        out = render_fn_inner(params, avals, bvals, key, rays[:,i:i+chunk], rand, True)
        if i==0:
            rets = out
        else:
            rets = [np.concatenate([a, b], 0) for a, b in zip(rets, out)]
    return rets

def make_network(num_layers, num_channels):
    layers = []
    for i in range(num_layers-1):
        layers.append(stax.Dense(num_channels))
        layers.append(stax.Relu)
    layers.append(stax.Dense(4))
    return stax.serial(*layers)

In [ ]:
def tree_numel(tree):
    return int(sum(x.size for x in jax.tree_util.tree_leaves(tree)))

def loss_fn(params, avals, bvals, key, rays, target, stratified):
    rgb = render_fn_inner(params, avals, bvals, key, rays, stratified, False)
    l = np.mean(np.square(rgb - target))
    return l

def train_model(lr, iters, avals, bvals, stratified, name='', plot_groups=None):
    rng = random.PRNGKey(0)
    if bvals is not None:
        init_shape = (-1, bvals.shape[0]*2)
    else:
        init_shape = (-1, 3)
    _, net_params = init_fn(rng, init_shape)

    opt_init, opt_update, get_params = optimizers.adam(lr)
    opt_state = opt_init(net_params)

    @jit
    def step_fn(i, opt_state, avals, bvals, key, rays, target):
        params = get_params(opt_state)
        g = grad(loss_fn)(params, avals, bvals, key, rays, target, stratified)
        return opt_update(i, g, opt_state)

    if plot_groups is not None:
        plot_groups['PSNR'].append(f'{name}')
    b_i = 0
    xs = []
    psnrs = []
    import time
    t = time.time()
    t0 = t
    for i in range(iters+1):
        batch = training_data[b_i:b_i+batch_size]
        b_i += batch_size
        rays = np.moveaxis(batch[:,:2], 1, 0)
        target = batch[:,2]
        if b_i >= training_data.shape[0]:
            b_i = 0

        rng, key = random.split(rng)
        opt_state = step_fn(i, opt_state, avals, bvals, key, rays, target)  
        
        if i%1000==0 or i==iters:
            psnr = []
            print(i, (time.time() - t) / 200, 'secs per iter', (time.time()-t0)/60., 'total mins')
            num_vals = val_poses.shape[0] if i==iters else 1
            for v in range(num_vals):
                # Render the holdout view for logging
                rays = get_rays(H, W, focal, val_poses[v,...])
                rng, key = random.split(rng)
                rgb, depth, acc = render_fn(get_params(opt_state), avals, bvals, key, rays, False)
                
                loss = np.mean(np.square(rgb - val_images[v,...]))
                psnr.append(-10. * np.log10(loss))
            psnr = np.mean(np.array(psnr))
            psnrs.append(psnr)
            xs.append(i)
            if plot_groups is not None:
                plotlosses_model.update({f'{name}':psnr}, current_step=i)
                plotlosses_model.send()
            t = time.time()

    best_state = jax.device_get(get_params(opt_state))
    trainable_params = tree_numel(best_state)
    encoding_params = int(bvals.size + avals.size)
    print(f'Number of parameters: {trainable_params + encoding_params}')

    results = {
        'state': get_params(opt_state),
        'psnrs': psnrs,
        'avals': avals,
        'bvals': bvals,
        'val_image': rgb,
        'xs': xs
    }
    return results

In [ ]:
# varying width
model_params = {
    'mapping_size': 256,
    'width': 512,
    'num_layers': 4,
    'scale': 8.,
}

compute_model_size('ffn_eta', model_params, n_dims=3)

In [ ]:
live_plot = True
reset_plots = True

training_steps = 50000

lr =  5e-4

batch_size = 1024
N_samples = 128
num_layers =  4
stratified_sampling = True

embedding_size = 512
gaussian_scale = 8


param_dict = {}

bvals = random.normal(rng, (embedding_size, 3))
avals = np.ones((bvals.shape[0]))

if live_plot:
    if reset_plots:
        plt_groups = {'PSNR':[]}
        plotlosses_model = PlotLosses(groups=plt_groups)
else:
    plt_groups = None

if reset_plots:
    outputs_paper = {}

layer_widths = [192, 512]
for layer_width in tqdm(layer_widths, leave=False):
    init_fn, apply_fn = make_network(num_layers, layer_width)
    outputs_paper[f'{layer_width}'] = train_model(lr, training_steps, avals, bvals * gaussian_scale, stratified_sampling, name=layer_width, plot_groups=plt_groups)

In [ ]:
outputs = {}
for w in outputs_paper.keys():
    state = outputs_paper[w]['state']
    avals = outputs_paper[w]['avals']
    bvals = outputs_paper[w]['bvals']
    rays = get_rays(H, W, focal, test_poses[TEST_IDX,...])
    rng, key = random.split(rng)
    rgb, depth, acc = render_fn(state, avals, bvals, key, rays, False)
    
    outputs[w] = {}
    outputs[w]['best_pred'] = rgb

with open(f"3d_nerf/ffn.pkl", "wb") as f:
    pickle.dump(outputs, f)

plot_error_heatmaps(test_images[TEST_IDX,...], outputs, model_name="FFN")